<div align="center">
  <img src="assets/Day12.png" alt="Databricks 14 Days AI Challenge - Day 12" width="800"/>
</div>

## DAY 12 (20/01/26) – MLflow Basics

### 📚 Learning Objectives
Training a model is easy. Keeping track of *which* model worked best is hard. Today we learn **MLOps** (Machine Learning Operations) using **MLflow**, the industry standard for managing the ML lifecycle.
* **Experiment Tracking:** Logging every parameter (e.g., "Learning Rate") and metric (e.g., "Accuracy").
* **Model Registry:** A central repository to store versioned models. 
* **Reproducibility:** Ensuring we can recreate the exact results 6 months from now.

### 🚀 Strategy: "The Experiment Log"
1.  **Data Prep:** Load our aggregated **Gold** data (Product Performance) and convert it to Pandas (since it's small and aggregated).
2.  **Experiment 1 (Baseline):** Train a simple **Linear Regression** to predict *Purchases* based on *Views*.
3.  **Experiment 2 (Challenger):** Train a **Random Forest** model to see if complexity improves performance.
4.  **Compare:** Use the MLflow UI to compare the two runs side-by-side.

###Data Loading & Preparation
**Task**: Load Gold data and prepare for Scikit-Learn. **Concept**: Spark is for Big Data processing. Scikit-Learn is for in-memory ML. Since our Gold table is already aggregated (one row per product), it fits easily into memory, so we convert it to Pandas.

In [0]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Setup Context
catalog = "course_catalog"
schema = "ecommerce_governed"
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

# 1. Load Data from Gold Layer
# We use the product performance table we built in Day 6/9
print("⏳ Loading Gold Data into Memory...")
df_gold = spark.table("gold_product_perf").toPandas()

# 2. Feature Selection
# Task: Predict 'unique_purchases' (y) based on 'unique_views' (X)
# We fill NaNs with 0 to prevent sklearn errors
df_clean = df_gold.fillna(0)

X = df_clean[["unique_views"]]
y = df_clean["unique_purchases"]

# 3. Train/Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"✅ Data Prepared. Training samples: {len(X_train)}, Test samples: {len(X_test)}")
display(X_train.head(5))

###Experiment 1 - Linear Regression
**Task**: Train a baseline model and log it with MLflow. **Concept**: We use mlflow.start_run() to create a "container" for this experiment. Anything we log inside this block belongs to this specific run.

In [0]:
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

# Set Experiment Name (Organizes runs in the UI)
user = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
mlflow.set_experiment(f"/Users/{user}/Day12_Product_Prediction")

print("🚀 Starting Run 1: Linear Regression (Baseline)...")

with mlflow.start_run(run_name="Run_1_Linear_Regression"):
    # 1. Log Parameters (Configuration)
    model_type = "Linear Regression"
    mlflow.log_param("model_type", model_type)
    mlflow.log_param("features", "unique_views")
    
    # 2. Train Model
    lr = LinearRegression()
    lr.fit(X_train, y_train)
    
    # 3. Predict & Evaluate
    predictions = lr.predict(X_test)
    r2 = r2_score(y_test, predictions)
    mse = mean_squared_error(y_test, predictions)
    
    # 4. Log Metrics (Performance)
    print(f"   📊 R2 Score: {r2:.4f}")
    mlflow.log_metric("r2_score", r2)
    mlflow.log_metric("mse", mse)
    
    # 5. Log Model Artifact (The actual file)
    # This allows us to load it later for deployment
    mlflow.sklearn.log_model(lr, "model")
    
    print(f"✅ Run Complete. Experiment ID: {mlflow.active_run().info.experiment_id}")

###Experiment 2 - Random Forest (Comparison)
**Task**: Train a more complex model to compare results. **Concept**: In data science, you rarely get it right on the first try. MLflow helps us compare "Apples to Apples" to see if the extra complexity of a Random Forest is worth it.

In [0]:
from sklearn.ensemble import RandomForestRegressor

print("🚀 Starting Run 2: Random Forest (Challenger)...")

with mlflow.start_run(run_name="Run_2_Random_Forest"):
    # 1. Log Parameters
    n_estimators = 100
    max_depth = 5
    
    mlflow.log_param("model_type", "Random Forest Regressor")
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)
    
    # 2. Train Model
    rf = RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    rf.fit(X_train, y_train)
    
    # 3. Evaluate
    predictions = rf.predict(X_test)
    r2 = r2_score(y_test, predictions)
    mse = mean_squared_error(y_test, predictions)
    
    # 4. Log Metrics
    print(f"   📊 R2 Score: {r2:.4f}")
    mlflow.log_metric("r2_score", r2)
    mlflow.log_metric("mse", mse)
    
    # 5. Log Model
    mlflow.sklearn.log_model(rf, "model")
    
    print("✅ Run Complete.")

### 🕵️‍♂️ Analyzing Results in the MLflow UI

We have now logged two different models. To visually compare them:

1.  **Click "Experiments"** in the top-right sidebar of this notebook (or the Beaker icon in the left nav bar).
2.  You will see a list of runs (`Run_1_Linear_Regression` and `Run_2_Random_Forest`).
3.  **Click the External Link icon** (arrow pointing out of a box) to open the full MLflow UI.
4.  **Select both runs** using the checkboxes.
5.  Click **"Compare"**.

**What to look for:**
* **Scatter Plot:** X-axis = `r2_score`. You want the model furthest to the right (closest to 1.0).
* **Parameters:** See which settings produced the winning score.

<div align="center">
  <img src="assets/day12_visuals.png" alt="Databricks 14 Days AI Challenge - Day 12" width="800"/>

### 🧠 Key Learnings & Takeaways
* **Tracking > Remembering:** We didn't have to write down "Run 1 had R2 of 0.85" on a piece of paper. MLflow stored it automatically.
* **Artifact Logging:** We saved not just the *score*, but the *actual model file* (`model.pkl`). This means we can deploy the best model immediately without retraining.
* **Hybrid Workflow:** We used **Spark** for the heavy lifting (ETL) and **Pandas/Scikit-Learn** for the modeling, bridging the gap between Data Engineering and Data Science.